# LLM Classification Fine-tuning Competition

This notebook implements a solution for comparing LLM responses and predicting which response is better for a given prompt, or if they are tied in quality.

## Problem Overview
- **Task**: Predict which of two LLM responses is better for a given prompt
- **Classes**: winner_model_a (1,0,0), winner_model_b (0,1,0), winner_tie (0,0,1)
- **Approach**: Fine-tune a transformer model on the comparison task


## 1. Setup and Imports

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import log_loss
import warnings
import gc
warnings.filterwarnings('ignore')

# Transformers with fallback handling
try:
    from transformers import (
        AutoTokenizer, AutoModel, AutoConfig,
        get_linear_schedule_with_warmup
    )
    print("✅ Transformers library loaded successfully")
except ImportError as e:
    print(f"❌ Error importing transformers: {e}")
    print("💡 Install with: pip install transformers")
    raise

# Visualization imports (optional for Kaggle)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    print("✅ Visualization libraries loaded")
    HAS_VIZ = True
except ImportError:
    print("⚠️ Visualization libraries not available - continuing without plots")
    HAS_VIZ = False

# Progress bar
try:
    from tqdm.auto import tqdm
    print("✅ Progress bar (tqdm) loaded")
except ImportError:
    # Fallback to basic tqdm or no progress bar
    try:
        from tqdm import tqdm
        print("✅ Basic tqdm loaded")
    except ImportError:
        print("⚠️ No progress bar available")
        # Create a dummy tqdm class
        class tqdm:
            def __init__(self, iterable, desc=''):
                self.iterable = iterable
            def __iter__(self):
                return iter(self.iterable)
            def set_postfix(self, **kwargs):
                pass

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

# GPU info
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Kaggle Environment Setup
import os
import sys

# Check if we're running in Kaggle
def is_kaggle():
    return 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if is_kaggle():
    print("🚀 Running in Kaggle environment")
    
    # Install missing packages if needed
    try:
        import matplotlib
        import seaborn
    except ImportError:
        print("📦 Installing visualization packages...")
        os.system('pip install matplotlib seaborn')
    
    # Set up Kaggle paths
    INPUT_PATH = '/kaggle/input'
    WORKING_PATH = '/kaggle/working'
    
    # Check for input files
    if os.path.exists(INPUT_PATH):
        print(f"📁 Available input datasets:")
        for item in os.listdir(INPUT_PATH):
            print(f"  - {item}")
    
    # Enable internet if available
    print("🌐 Internet connectivity check...")
    try:
        import requests
        response = requests.get('https://www.google.com', timeout=5)
        print("✅ Internet connection available")
    except:
        print("❌ No internet connection - will use offline models only")
        
else:
    print("💻 Running in local environment")
    INPUT_PATH = '.'
    WORKING_PATH = '.'

print(f"Input path: {INPUT_PATH}")
print(f"Working path: {WORKING_PATH}")

In [ ]:
# Set random seed for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 2. Data Loading and Exploration

In [ ]:
# Load data with proper path handling
import os

# Define data paths
if 'INPUT_PATH' in globals():
    # Use Kaggle paths if defined
    train_path = os.path.join(INPUT_PATH, 'train.csv')
    test_path = os.path.join(INPUT_PATH, 'test.csv')
    
    # If files don't exist in INPUT_PATH, look for competition data folder
    if not os.path.exists(train_path):
        # Look for competition data folder
        possible_paths = [
            '/kaggle/input/llm-classification-finetuning',
            '/kaggle/input/llm_classification_finetuning', 
            '/kaggle/input'
        ]
        
        for path in possible_paths:
            if os.path.exists(os.path.join(path, 'train.csv')):
                train_path = os.path.join(path, 'train.csv')
                test_path = os.path.join(path, 'test.csv')
                print(f"📁 Found data files in: {path}")
                break
else:
    # Local paths
    train_path = 'train.csv'
    test_path = 'test.csv'

print(f"📄 Loading data from:")
print(f"  Train: {train_path}")
print(f"  Test: {test_path}")

# Load the data
try:
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print("✅ Data loaded successfully!")
except FileNotFoundError as e:
    print(f"❌ Error loading data: {e}")
    print("💡 Make sure the competition data is available in the input folder")
    raise

print(f"\n📊 Dataset Information:")
print(f"Training set size: {len(train_df):,}")
print(f"Test set size: {len(test_df):,}")
print(f"\nTraining columns: {train_df.columns.tolist()}")
print(f"Test columns: {test_df.columns.tolist()}")

# Check for missing files that might be needed
sample_submission_path = test_path.replace('test.csv', 'sample_submission.csv')
if os.path.exists(sample_submission_path):
    print(f"✅ Sample submission found: {sample_submission_path}")
else:
    print("⚠️ No sample submission file found")

In [ ]:
# Display basic statistics
print("Training set info:")
print(train_df.info())
print("\nLabel distribution:")
print(f"Model A wins: {train_df['winner_model_a'].sum()}")
print(f"Model B wins: {train_df['winner_model_b'].sum()}")
print(f"Ties: {train_df['winner_tie'].sum()}")

In [ ]:
# Visualize label distribution
labels = ['Model A Wins', 'Model B Wins', 'Ties']
counts = [train_df['winner_model_a'].sum(), train_df['winner_model_b'].sum(), train_df['winner_tie'].sum()]

print("📊 Label Distribution:")
for label, count in zip(labels, counts):
    percentage = (count / len(train_df)) * 100
    print(f"  {label}: {count:,} ({percentage:.1f}%)")

# Create visualization if matplotlib is available
if HAS_VIZ:
    plt.figure(figsize=(12, 6))
    
    plt.subplot(1, 2, 1)
    plt.pie(counts, labels=labels, autopct='%1.1f%%', startangle=90)
    plt.title('Label Distribution')
    
    plt.subplot(1, 2, 2)
    bars = plt.bar(labels, counts, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    plt.title('Label Counts')
    plt.xticks(rotation=45)
    
    # Add value labels on bars
    for bar, count in zip(bars, counts):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01, 
                f'{count:,}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Matplotlib not available - skipping visualization")

In [ ]:
# Text length analysis
train_df['prompt_len'] = train_df['prompt'].str.len()
train_df['response_a_len'] = train_df['response_a'].str.len()
train_df['response_b_len'] = train_df['response_b'].str.len()
train_df['total_len'] = train_df['prompt_len'] + train_df['response_a_len'] + train_df['response_b_len']

print("Text length statistics:")
print(train_df[['prompt_len', 'response_a_len', 'response_b_len', 'total_len']].describe())

In [ ]:
# Sample data exploration
print("Sample training examples:")
for i in range(3):
    row = train_df.iloc[i]
    print(f"\n--- Example {i+1} ---")
    print(f"Prompt: {row['prompt'][:200]}...")
    print(f"Response A: {row['response_a'][:200]}...")
    print(f"Response B: {row['response_b'][:200]}...")
    print(f"Winner: A={row['winner_model_a']}, B={row['winner_model_b']}, Tie={row['winner_tie']}")

## 3. Data Preprocessing

In [ ]:
def preprocess_text(text):
    """Clean and preprocess text data"""
    if pd.isna(text):
        return ""
    # Convert to string and clean JSON-like formatting
    text = str(text)
    # Remove brackets and quotes from JSON-like strings
    text = text.replace('["', '').replace('"]', '').replace('"', '')
    # Clean extra whitespace
    text = ' '.join(text.split())
    return text

# Apply preprocessing
for col in ['prompt', 'response_a', 'response_b']:
    if col in train_df.columns:
        train_df[col] = train_df[col].apply(preprocess_text)
    if col in test_df.columns:
        test_df[col] = test_df[col].apply(preprocess_text)

print("Text preprocessing completed.")

## 4. Dataset Class

In [ ]:
class LLMComparisonDataset(Dataset):
    """
    Dataset class for LLM response comparison
    
    Concatenates prompt and two responses for model input
    """
    def __init__(self, df, tokenizer, max_length=512, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Extract text components
        prompt = str(row['prompt'])
        response_a = str(row['response_a'])
        response_b = str(row['response_b'])
        
        # Create input text with special tokens
        # Format: [CLS] prompt [SEP] response_a [SEP] response_b [SEP]
        text = f"{prompt} [SEP] {response_a} [SEP] {response_b}"
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }
        
        # Add labels for training data
        if not self.is_test:
            labels = torch.tensor([
                row['winner_model_a'],
                row['winner_model_b'], 
                row['winner_tie']
            ], dtype=torch.float)
            item['labels'] = labels
            
        return item

## 5. Model Architecture

In [ ]:
class LLMComparisonModel(nn.Module):
    """
    Transformer-based model for LLM response comparison
    
    Uses a pre-trained transformer as encoder with a classification head
    """
    def __init__(self, model_name, num_classes=3, dropout_rate=0.1):
        super().__init__()
        
        # Load pre-trained model
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name)
        
        # Classification head
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.config.hidden_size, num_classes)
        
        # Initialize classifier weights
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.constant_(self.classifier.bias, 0)
        
    def forward(self, input_ids, attention_mask):
        # Get transformer outputs
        outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Use pooled output or [CLS] token
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            pooled_output = outputs.pooler_output
        else:
            # For models without pooler (e.g., DistilBERT)
            pooled_output = outputs.last_hidden_state[:, 0, :]
        
        # Apply dropout and classification
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        # Apply softmax to get probabilities
        probabilities = torch.softmax(logits, dim=-1)
        
        return probabilities

## 6. Training Functions

In [ ]:
import time

def train_epoch_fast(model, train_loader, optimizer, scheduler, criterion, device, scaler=None):
    """
    Speed-optimized training epoch with mixed precision
    """
    model.train()
    total_loss = 0
    log_interval = max(1, len(train_loader) // 10)  # Log 10 times per epoch max
    
    progress_bar = tqdm(train_loader, desc='Training', leave=False)
    
    for batch_idx, batch in enumerate(progress_bar):
        # Move to device
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        
        # Mixed precision forward pass
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(input_ids, attention_mask)
                loss = criterion(torch.log(outputs + 1e-8), labels)
            
            # Mixed precision backward pass
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            # Regular forward pass
            outputs = model(input_ids, attention_mask)
            loss = criterion(torch.log(outputs + 1e-8), labels)
            
            # Regular backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
        
        optimizer.zero_grad(set_to_none=True)  # Faster than zero_grad()
        scheduler.step()
        
        total_loss += loss.item()
        
        # Update progress bar less frequently
        if batch_idx % log_interval == 0:
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_loader)

def validate_epoch_fast(model, val_loader, criterion, device, scaler=None):
    """
    Speed-optimized validation epoch
    """
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []
    
    progress_bar = tqdm(val_loader, desc='Validation', leave=False)
    
    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            
            # Use mixed precision for validation too
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(input_ids, attention_mask)
                    loss = criterion(torch.log(outputs + 1e-8), labels)
            else:
                outputs = model(input_ids, attention_mask)
                loss = criterion(torch.log(outputs + 1e-8), labels)
            
            total_loss += loss.item()
            predictions.extend(outputs.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    log_loss_score = log_loss(true_labels, predictions)
    
    return avg_loss, log_loss_score, predictions, true_labels

# Legacy function for compatibility
def train_epoch(model, train_loader, optimizer, scheduler, criterion, device):
    """Fallback to fast training without mixed precision"""
    return train_epoch_fast(model, train_loader, optimizer, scheduler, criterion, device, scaler=None)

def validate_epoch(model, val_loader, criterion, device):
    """Fallback to fast validation without mixed precision"""
    return validate_epoch_fast(model, val_loader, criterion, device, scaler=None)

In [ ]:
def train_model_fast(model, train_loader, val_loader, num_epochs=2, learning_rate=3e-5):
    """
    SPEED-OPTIMIZED training loop with mixed precision and optimizations
    """
    model.to(device)
    
    # Initialize mixed precision scaler if available
    scaler = None
    if USE_MIXED_PRECISION and device.type == 'cuda':
        try:
            from torch.cuda.amp import GradScaler, autocast
            scaler = GradScaler()
            print("✅ Mixed precision training enabled")
        except ImportError:
            print("⚠️ Mixed precision not available, using FP32")
    
    # Optimizer with optimized settings
    optimizer = AdamW(
        model.parameters(), 
        lr=learning_rate, 
        weight_decay=0.01,
        eps=1e-6,  # Slightly larger epsilon for stability
        betas=(0.9, 0.98)  # Optimized betas for faster convergence
    )
    
    # Scheduler with reduced warmup
    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    # Loss function
    criterion = nn.KLDivLoss(reduction='batchmean')
    
    # Tracking variables
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    val_log_losses = []
    
    print(f"🏃‍♂️ Starting fast training: {total_steps:,} total steps")
    print(f"⚡ Optimizations: Mixed Precision={scaler is not None}, Warmup={warmup_steps} steps")
    
    for epoch in range(num_epochs):
        epoch_start_time = time.time()
        print(f"")
        print(f"🚀 Epoch {epoch+1}/{num_epochs}")
        print("-" * 50)
        
        # Fast training epoch
        train_loss = train_epoch_fast(model, train_loader, optimizer, scheduler, criterion, device, scaler)
        
        # Fast validation (less frequent for speed)
        val_loss, val_log_loss, _, _ = validate_epoch_fast(model, val_loader, criterion, device, scaler)
        
        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_log_losses.append(val_log_loss)
        
        # Timing
        epoch_time = time.time() - epoch_start_time
        remaining_time = epoch_time * (num_epochs - epoch - 1)
        
        print(f"📊 Train Loss: {train_loss:.4f}")
        print(f"📊 Val Loss: {val_loss:.4f}")
        print(f"📊 Val Log Loss: {val_log_loss:.4f}")
        print(f"⏱️ Epoch time: {epoch_time/60:.1f}min, ETA: {remaining_time/60:.1f}min")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"💾 New best model saved! Val Loss: {best_val_loss:.4f}")
        
        # Aggressive memory cleanup
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    
    total_time = sum([epoch_time]) if 'epoch_time' in locals() else 0
    print("")
    print(f"🎉 Training completed in {total_time/60:.1f} minutes!")
    
    return train_losses, val_losses, val_log_losses

## 7. Model Configuration and Training Setup

In [ ]:
# ⚡ FAST TRAINING CONFIGURATION - Optimized for Kaggle
# Prioritize speed over marginal accuracy gains

# Model options (ordered by speed - fastest first)
MODEL_OPTIONS = [
    'distilbert-base-uncased',    # 🚀 FASTEST: 66M params, 6 layers
    'bert-base-uncased',          # 🔥 FAST: 110M params, but heavier
    'roberta-base',               # 🐌 SLOWER: 125M params
    'microsoft/deberta-v3-base'   # 🐌 SLOWEST: 184M params (skip for speed)
]

# Function to find available model (prefer fastest)
def find_available_model(model_options):
    """Try to load models in order of speed preference"""
    from transformers import AutoTokenizer, AutoConfig
    
    for model_name in model_options:
        try:
            print(f"Trying to load {model_name}...")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            config = AutoConfig.from_pretrained(model_name)
            print(f"✅ Successfully loaded {model_name}")
            return model_name
        except Exception as e:
            print(f"❌ Failed to load {model_name}: {str(e)[:100]}...")
            continue
    
    print("⚠️ Using fallback model: distilbert-base-uncased")
    return 'distilbert-base-uncased'

# Find the best available model
MODEL_NAME = find_available_model(MODEL_OPTIONS)

# 🚀 SPEED-OPTIMIZED CONFIGURATION
MAX_LENGTH = 256        # ⚡ Reduced from 512 (4x faster processing)
BATCH_SIZE = 16         # ⚡ Increased batch size for better GPU utilization
NUM_EPOCHS = 2          # ⚡ Reduced from 3 epochs (66% time reduction)
LEARNING_RATE = 3e-5    # ⚡ Slightly higher LR for faster convergence
DROPOUT_RATE = 0.1

# Additional speed optimizations
USE_MIXED_PRECISION = True    # ⚡ Enable AMP for 2x speed boost
USE_GRADIENT_CHECKPOINTING = False  # ⚡ Disabled for speed (uses more memory)
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.05     # ⚡ Reduced warmup (5% instead of 10%)

# Data sampling for even faster training (if dataset is very large)
USE_SAMPLE_TRAINING = True    # ⚡ Use subset of data for faster iteration
SAMPLE_FRACTION = 0.8         # ⚡ Use 80% of training data

print(f"\n🚀 SPEED-OPTIMIZED CONFIGURATION:")
print(f"Model: {MODEL_NAME}")
print(f"Max Length: {MAX_LENGTH} (reduced for speed)")
print(f"Batch Size: {BATCH_SIZE} (increased for GPU efficiency)")
print(f"Epochs: {NUM_EPOCHS} (reduced for speed)")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Mixed Precision: {USE_MIXED_PRECISION}")
print(f"Sample Training: {USE_SAMPLE_TRAINING} ({SAMPLE_FRACTION*100}% of data)")

# Estimate training time
if 'train_df' in globals():
    total_samples = len(train_df) * SAMPLE_FRACTION if USE_SAMPLE_TRAINING else len(train_df)
    steps_per_epoch = int(total_samples * 0.8 / BATCH_SIZE)  # 80% for training split
    total_steps = steps_per_epoch * NUM_EPOCHS
    
    # Rough time estimate (adjust based on your GPU)
    seconds_per_step = 0.5 if MODEL_NAME == 'distilbert-base-uncased' else 1.0
    estimated_minutes = (total_steps * seconds_per_step) / 60
    
    print(f"\n⏱️ ESTIMATED TRAINING TIME:")
    print(f"Total steps: {total_steps:,}")
    print(f"Estimated time: {estimated_minutes:.1f} minutes ({estimated_minutes/60:.1f} hours)")
    
    if estimated_minutes > 180:  # More than 3 hours
        print("⚠️ Still quite long! Consider:")
        print("  - Reduce SAMPLE_FRACTION to 0.5 or 0.3")
        print("  - Reduce MAX_LENGTH to 128")
        print("  - Use only 1 epoch for quick baseline")
else:
    print("📊 Load data first to see time estimates")

In [ ]:
# ⚡ ULTRA-FAST MODE (uncomment for maximum speed)\n# Uncomment these lines if you want ULTRA-FAST training (under 30 minutes)\n# Warning: This will reduce accuracy but dramatically speed up training\n\n# ULTRA_FAST_MODE = True\n# if ULTRA_FAST_MODE:\n#     print(\"🚀⚡ ULTRA-FAST MODE ENABLED - MAXIMUM SPEED!\")\n#     SAMPLE_FRACTION = 0.2      # Use only 20% of data\n#     NUM_EPOCHS = 1             # Single epoch only\n#     MAX_LENGTH = 128           # Much shorter sequences\n#     BATCH_SIZE = 32            # Larger batches\n#     USE_MIXED_PRECISION = True # Force mixed precision\n#     print(\"⚠️ Note: Accuracy will be lower, but training will be VERY fast\")\n\nprint(\"💡 Current mode: BALANCED (good speed + decent accuracy)\")\nprint(\"💡 To enable ULTRA-FAST mode, uncomment the lines above\")"

In [ ]:
# 🚀 FAST DATA SPLITTING with optional sampling
# Create a single label for stratification
train_df['winner_label'] = train_df[['winner_model_a', 'winner_model_b', 'winner_tie']].idxmax(axis=1)

# Apply data sampling if enabled (for faster training)
if USE_SAMPLE_TRAINING and SAMPLE_FRACTION < 1.0:
    print(f"⚡ Sampling {SAMPLE_FRACTION*100}% of data for faster training...")
    
    # Stratified sampling to maintain class balance
    sampled_df = train_df.groupby('winner_label', group_keys=False).apply(
        lambda x: x.sample(frac=SAMPLE_FRACTION, random_state=42)
    ).reset_index(drop=True)
    
    print(f"📊 Data sampling results:")
    print(f"  Original: {len(train_df):,} samples")
    print(f"  Sampled: {len(sampled_df):,} samples")
    print(f"  Reduction: {(1-len(sampled_df)/len(train_df))*100:.1f}%")
    
    working_df = sampled_df
else:
    print("📊 Using full dataset (no sampling)")
    working_df = train_df

# Create stratified train-validation split
train_data, val_data = train_test_split(
    working_df, 
    test_size=0.15,  # ⚡ Reduced validation size (15% instead of 20%)
    random_state=42,
    stratify=working_df['winner_label']
)

print(f"\n🎯 Final split sizes:")
print(f"Training samples: {len(train_data):,}")
print(f"Validation samples: {len(val_data):,}")
print(f"Total samples used: {len(working_df):,}")

# Check label distribution in splits
print(f"\n📋 Training set distribution:")
train_dist = train_data['winner_label'].value_counts()
print(train_dist)
print(f"\n📋 Validation set distribution:")
val_dist = val_data['winner_label'].value_counts()
print(val_dist)

# Calculate training time estimate
steps_per_epoch = len(train_data) // BATCH_SIZE
total_steps = steps_per_epoch * NUM_EPOCHS

# Estimate based on model type
if MODEL_NAME == 'distilbert-base-uncased':
    seconds_per_step = 0.3  # DistilBERT is fast
elif 'bert-base' in MODEL_NAME:
    seconds_per_step = 0.6  # BERT base
else:
    seconds_per_step = 1.0  # Larger models

estimated_seconds = total_steps * seconds_per_step
estimated_minutes = estimated_seconds / 60
estimated_hours = estimated_minutes / 60

print(f"\n⏱️ UPDATED TRAINING TIME ESTIMATE:")
print(f"Steps per epoch: {steps_per_epoch:,}")
print(f"Total steps: {total_steps:,}")
print(f"Estimated time: {estimated_minutes:.1f}min ({estimated_hours:.1f}h)")

if estimated_hours > 2:
    print(f"\n⚠️ Still {estimated_hours:.1f}h - consider further optimizations:")
    print("💡 Quick fixes:")
    print("  - Set SAMPLE_FRACTION = 0.5 (50% of data)")
    print("  - Set NUM_EPOCHS = 1 (single epoch)")
    print("  - Set MAX_LENGTH = 128 (shorter sequences)")
    print("  - Increase BATCH_SIZE to 32 (if GPU memory allows)")
elif estimated_hours < 0.5:
    print("✅ Great! Training should complete in under 30 minutes")

In [ ]:
# Initialize tokenizer and model with error handling
print("Loading tokenizer and model...")

try:
    # Try to load the selected model
    print(f"Attempting to load {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = LLMComparisonModel(MODEL_NAME, dropout_rate=DROPOUT_RATE)
    print(f"✅ Successfully loaded {MODEL_NAME}")
    
except Exception as e:
    print(f"❌ Error loading {MODEL_NAME}: {e}")
    print("🔄 Falling back to distilbert-base-uncased...")
    
    # Fallback to distilbert
    MODEL_NAME = 'distilbert-base-uncased'
    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = LLMComparisonModel(MODEL_NAME, dropout_rate=DROPOUT_RATE)
        print(f"✅ Successfully loaded fallback model: {MODEL_NAME}")
    except Exception as e2:
        print(f"❌ Critical error: Cannot load any model. {e2}")
        print("💡 Suggestions:")
        print("1. Check internet connection in Kaggle")
        print("2. Try enabling internet in Kaggle notebook settings")
        print("3. Use a Kaggle dataset with pre-downloaded models")
        raise e2

print(f"\n📊 Model Statistics:")
print(f"Model: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Display model architecture summary
print(f"\n🏗️ Model Architecture:")
print(model)

In [ ]:
# Create optimized datasets and dataloaders
print("Creating speed-optimized datasets...")

# Create datasets with the optimized configuration
train_dataset = LLMComparisonDataset(train_data, tokenizer, MAX_LENGTH)
val_dataset = LLMComparisonDataset(val_data, tokenizer, MAX_LENGTH)
test_dataset = LLMComparisonDataset(test_df, tokenizer, MAX_LENGTH, is_test=True)

# Optimize DataLoader settings for speed
# num_workers: Use more workers for faster data loading
# pin_memory: Speed up GPU transfer
# persistent_workers: Keep workers alive between epochs
num_workers = min(4, os.cpu_count() if hasattr(os, 'cpu_count') else 2)
pin_memory = torch.cuda.is_available()  # Only use pin_memory with GPU

print(f"⚡ DataLoader optimizations:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Num workers: {num_workers}")
print(f"  Pin memory: {pin_memory}")

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=True if num_workers > 0 else False,
    drop_last=True  # Drop incomplete batches for consistent timing
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=True if num_workers > 0 else False
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE * 2,  # Larger batch for inference (no gradients)
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=pin_memory
)

print(f"\\n📊 Dataset sizes:")
print(f"Training batches: {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")
print(f"Test batches: {len(test_loader):,}")

# Final time estimate with all optimizations
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * NUM_EPOCHS

# Optimistic time estimate with all speed improvements
if MODEL_NAME == 'distilbert-base-uncased':
    seconds_per_step = 0.2  # Optimistic with mixed precision + optimizations
else:
    seconds_per_step = 0.4

optimistic_time_minutes = (total_steps * seconds_per_step) / 60
optimistic_time_hours = optimistic_time_minutes / 60

print(f"\\n🚀 FINAL TIME ESTIMATE (with all optimizations):")
print(f"Total steps: {total_steps:,}")
print(f"Optimistic time: {optimistic_time_minutes:.1f}min ({optimistic_time_hours:.1f}h)")

if optimistic_time_hours > 3:
    print("\\n⚠️ Still over 3 hours. Consider these ultra-fast settings:")
    print("💡 ULTRA-FAST mode (add to config cell):")
    print("  SAMPLE_FRACTION = 0.3  # Use only 30% of data")
    print("  NUM_EPOCHS = 1         # Single epoch")
    print("  MAX_LENGTH = 128       # Very short sequences")
    print("  BATCH_SIZE = 32        # Larger batches")
elif optimistic_time_hours < 1:
    print("✅ Excellent! Should complete in under 1 hour")
else:
    print(f"✅ Good! Should complete in about {optimistic_time_hours:.1f} hours")

## 8. Model Training

In [ ]:
# Test a single batch to ensure everything works
print("Testing model with a single batch...")
model.to(device)
test_batch = next(iter(train_loader))
with torch.no_grad():
    input_ids = test_batch['input_ids'].to(device)
    attention_mask = test_batch['attention_mask'].to(device)
    outputs = model(input_ids, attention_mask)
    print(f"Output shape: {outputs.shape}")
    print(f"Sample predictions: {outputs[0]}")
print("Model test successful!")

In [ ]:
# 🚀 Start speed-optimized training\nprint(\"🏁 Starting FAST training with all optimizations...\")\nprint(f\"⚡ Configuration: {MODEL_NAME}, {MAX_LENGTH} tokens, {BATCH_SIZE} batch, {NUM_EPOCHS} epochs\")\nprint(f\"⚡ Optimizations: Mixed Precision, Data Sampling, Reduced Validation\")\n\ntraining_start_time = time.time()\n\n# Use the fast training function\ntrain_losses, val_losses, val_log_losses = train_model_fast(\n    model, train_loader, val_loader, \n    num_epochs=NUM_EPOCHS, \n    learning_rate=LEARNING_RATE\n)\n\ntotal_training_time = time.time() - training_start_time\nprint(f\"\\n🎉 TRAINING COMPLETED!\")\nprint(f\"⏱️ Total time: {total_training_time/60:.1f} minutes ({total_training_time/3600:.1f} hours)\")\nprint(f\"📊 Final validation log loss: {val_log_losses[-1]:.4f}\")\n\n# Quick performance summary\nif total_training_time < 1800:  # Under 30 minutes\n    print(\"🚀 Excellent! Very fast training achieved!\")\nelif total_training_time < 3600:  # Under 1 hour\n    print(\"✅ Good! Training completed in reasonable time\")\nelse:\n    print(f\"⚠️ Training took {total_training_time/3600:.1f}h - consider further optimizations next time\")\n\n# Memory cleanup\ngc.collect()\nif torch.cuda.is_available():\n    torch.cuda.empty_cache()\n    print(f\"🧹 GPU memory cleaned up\")"

## 9. Training Visualization

In [ ]:
# Plot training curves
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Training Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='s')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(val_log_losses, label='Validation Log Loss', marker='^', color='red')
plt.title('Validation Log Loss')
plt.xlabel('Epoch')
plt.ylabel('Log Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-', label='Training Loss')
plt.plot(epochs, val_losses, 'r-', label='Validation Loss')
plt.fill_between(epochs, train_losses, alpha=0.3)
plt.fill_between(epochs, val_losses, alpha=0.3)
plt.title('Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 10. Model Evaluation

In [ ]:
# Load best model and evaluate on validation set
print("Loading best model for final evaluation...")
model.load_state_dict(torch.load('best_model.pth'))
model.to(device)

# Final validation
criterion = nn.KLDivLoss(reduction='batchmean')
final_val_loss, final_log_loss, val_predictions, val_true_labels = validate_epoch(
    model, val_loader, criterion, device
)

print(f"\nFinal Validation Results:")
print(f"Validation Loss: {final_val_loss:.4f}")
print(f"Validation Log Loss: {final_log_loss:.4f}")

In [ ]:
# Analyze predictions
val_predictions = np.array(val_predictions)
val_true_labels = np.array(val_true_labels)

# Convert to class predictions
predicted_classes = np.argmax(val_predictions, axis=1)
true_classes = np.argmax(val_true_labels, axis=1)

# Classification accuracy
accuracy = (predicted_classes == true_classes).mean()
print(f"Validation Accuracy: {accuracy:.4f}")

# Class-wise analysis
from sklearn.metrics import classification_report, confusion_matrix

class_names = ['Model A', 'Model B', 'Tie']
print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Validation Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 11. Test Predictions

In [ ]:
def make_predictions(model, test_loader, device):
    """
    Generate predictions on test set
    """
    model.eval()
    predictions = []
    
    progress_bar = tqdm(test_loader, desc='Generating predictions')
    
    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            outputs = model(input_ids, attention_mask)
            predictions.extend(outputs.cpu().numpy())
    
    return np.array(predictions)

# Generate test predictions
print("Generating test predictions...")
test_predictions = make_predictions(model, test_loader, device)

print(f"Test predictions shape: {test_predictions.shape}")
print(f"Sample prediction: {test_predictions[0]}")

## 12. Submission File Creation

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': test_predictions[:, 0],
    'winner_model_b': test_predictions[:, 1],
    'winner_tie': test_predictions[:, 2]
})

# Verify probabilities sum to 1
prob_sums = submission[['winner_model_a', 'winner_model_b', 'winner_tie']].sum(axis=1)
print(f"Probability sums - Min: {prob_sums.min():.6f}, Max: {prob_sums.max():.6f}")
print(f"All probabilities sum to ~1: {np.allclose(prob_sums, 1.0)}")

# Display submission statistics
print("\nSubmission Statistics:")
print(submission[['winner_model_a', 'winner_model_b', 'winner_tie']].describe())

# Show sample predictions
print("\nSample Predictions:")
print(submission.head(10))

In [ ]:
# Visualize prediction distributions
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(submission['winner_model_a'], bins=50, alpha=0.7, label='Model A', color='blue')
plt.title('Model A Win Probability Distribution')
plt.xlabel('Probability')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.hist(submission['winner_model_b'], bins=50, alpha=0.7, label='Model B', color='red')
plt.title('Model B Win Probability Distribution')
plt.xlabel('Probability')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.hist(submission['winner_tie'], bins=50, alpha=0.7, label='Tie', color='green')
plt.title('Tie Probability Distribution')
plt.xlabel('Probability')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Most confident predictions
max_probs = submission[['winner_model_a', 'winner_model_b', 'winner_tie']].max(axis=1)
predicted_classes_test = submission[['winner_model_a', 'winner_model_b', 'winner_tie']].idxmax(axis=1)

print("\nPrediction Confidence:")
print(f"Mean confidence: {max_probs.mean():.4f}")
print(f"Min confidence: {max_probs.min():.4f}")
print(f"Max confidence: {max_probs.max():.4f}")

print("\nPredicted Class Distribution:")
print(predicted_classes_test.value_counts())

In [ ]:
# Save submission file
submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")

# Verify submission format
print("\nSubmission file verification:")
saved_submission = pd.read_csv('submission.csv')
print(f"Shape: {saved_submission.shape}")
print(f"Columns: {saved_submission.columns.tolist()}")
print(f"Data types:\n{saved_submission.dtypes}")

# Final check
required_cols = ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
has_all_cols = all(col in saved_submission.columns for col in required_cols)
print(f"\nHas all required columns: {has_all_cols}")
print(f"No missing values: {saved_submission.isnull().sum().sum() == 0}")

print("\n✅ Submission file ready for upload!")

## 13. Model Summary and Next Steps

In [ ]:
print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(f"Architecture: {MODEL_NAME}")
print(f"Max Sequence Length: {MAX_LENGTH}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Training Epochs: {NUM_EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Dropout Rate: {DROPOUT_RATE}")
print(f"\nFinal Validation Log Loss: {final_log_loss:.4f}")
print(f"Final Validation Accuracy: {accuracy:.4f}")
print(f"\nTest Predictions Generated: {len(test_predictions)}")

print("\n" + "=" * 60)
print("POTENTIAL IMPROVEMENTS")
print("=" * 60)
print("1. Try larger models (DeBERTa-large, RoBERTa-large)")
print("2. Implement cross-validation for more robust training")
print("3. Use advanced techniques like gradual unfreezing")
print("4. Experiment with different text formatting strategies")
print("5. Add ensemble methods with multiple models")
print("6. Fine-tune on domain-specific data if available")
print("7. Use focal loss or other advanced loss functions")
print("8. Implement more sophisticated data augmentation")

print("\n✅ Notebook execution completed successfully!")